# Data Preprocessing Pipeline

This notebook processes the training data by:
1. Extracting Value and Unit from catalog_content
2. Processing image links for OCR text extraction
3. Applying NLP for feature engineering
4. Creating a clean, processed DataFrame for modeling

In [28]:
# Import required libraries
import os
import pandas as pd
import numpy as np
import re
from pathlib import Path
import requests
from PIL import Image
import easyocr
from io import BytesIO
import warnings
warnings.filterwarnings('ignore')

# NLP libraries
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2

# Download required NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

# Initialize OCR reader
reader = easyocr.Reader(['en'])

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\vasishth\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\vasishth\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\vasishth\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


In [ ]:
# PaddleOCR setup
from paddleocr import PaddleOCR
ocr_engine = PaddleOCR(use_angle_cls=True, lang='en')

# OCR extraction function using PaddleOCR
def extract_text_paddleocr(url):
    try:
        response = requests.get(url, timeout=10)
        if response.status_code != 200:
            return ''
        img = Image.open(BytesIO(response.content)).convert('RGB')
        # Save to temp file for PaddleOCR
        temp_path = os.path.join(CACHE_DIR, 'temp_img.jpg')
        img.save(temp_path)
        result = ocr_engine.ocr(temp_path, cls=True)
        # Extract text lines
        text = ' '.join([line[1][0] for line in result[0]])
        text = re.sub(r'\s+', ' ', text).strip()
        return text
    except Exception as e:
        print(f"Error with PaddleOCR for {url}: {str(e)}")
        return ''

In [29]:
# Set up paths
SCRIPT_DIR = os.getcwd()
ROOT_DIR = os.path.dirname(SCRIPT_DIR)
DATASET_FOLDER = os.path.join(ROOT_DIR, 'dataset')
TRAIN_FILE = os.path.join(DATASET_FOLDER, 'sample_test.csv')
CACHE_DIR = os.path.join(DATASET_FOLDER, 'image_cache')
PROCESSED_OUT = os.path.join(DATASET_FOLDER, 'processed_for_model.csv')

# Create cache directory if it doesn't exist
os.makedirs(CACHE_DIR, exist_ok=True)

# Load training data
df = pd.read_csv(TRAIN_FILE)
print("Dataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nSample data:")
display(df.head())

Dataset shape: (100, 3)

Columns: ['sample_id', 'catalog_content', 'image_link']

Sample data:


,sample_id,catalog_content,image_link
0,217392,Item Name: Gift Basket Village Gourmet Meat an...,https://m.media-amazon.com/images/I/91GB1wC6Ob...
1,209156,"Item Name: NPG Dried Lotus Seeds 16 Oz, Uncook...",https://m.media-amazon.com/images/I/81VnzF1vkv...
2,262333,Item Name: Annies Homegrown Macaroni and Chees...,https://m.media-amazon.com/images/I/51aCDMHMnI...
3,295979,Item Name: Bear Creek Country Kitchens Creamy ...,https://m.media-amazon.com/images/I/71dzRyLGPi...
4,50604,Item Name: Japanese Kelp Kombu Umami Soup Stoc...,https://m.media-amazon.com/images/I/71Yu21cGwr...


In [30]:
# Function to extract Value and Unit from catalog_content
def extract_value_unit(content):
    if not isinstance(content, str):
        return pd.Series({'value': None, 'unit': ''})
    
    # Common units and their variations
    unit_patterns = {
        'kg': r'(?:kg|kgs|kilograms?)',
        'g': r'(?:g|grams?)',
        'l': r'(?:l|ltr|liters?|litres?)',
        'ml': r'(?:ml|milliliters?|millilitres?)',
        'cm': r'(?:cm|centimeters?|centimetres?)',
        'mm': r'(?:mm|millimeters?|millimetres?)',
        'in': r'(?:in|inch|inches)',
        'm': r'(?:m|meters?|metres?)',
        'oz': r'(?:oz|ounces?)',
        'lb': r'(?:lb|lbs|pounds?)'
    }
    
    # Build regex pattern for value and unit
    unit_pattern = '|'.join(f'({p})' for p in unit_patterns.values())
    value_pattern = r'(\d+(?:\.\d+)?(?:\s*-\s*\d+(?:\.\d+)?)?)'
    
    # Look for value followed by unit
    pattern = f'{value_pattern}\s*({unit_pattern})'
    matches = re.findall(pattern, content, re.IGNORECASE)
    
    if matches:
        value_str, *units = matches[0]
        # Get the first non-empty unit match
        unit = next((u for u in units if u), '')
        
        # Handle range values (take average)
        if '-' in value_str:
            start, end = map(float, value_str.split('-'))
            value = (start + end) / 2
        else:
            value = float(value_str)
            
        # Normalize unit
        unit = unit.lower().strip()
        for std_unit, patterns in unit_patterns.items():
            if re.match(f'^{patterns}$', unit, re.IGNORECASE):
                unit = std_unit
                break
                
        return pd.Series({'value': value, 'unit': unit})
    
    return pd.Series({'value': None, 'unit': ''})

# Apply extraction to catalog_content
extracted = df['catalog_content'].apply(extract_value_unit)
df[['value', 'unit']] = extracted

print("Value and Unit extraction complete.")
print("\nSample results:")
display(df[['catalog_content', 'value', 'unit']].head())

# Value statistics
print("\nValue statistics:")
print(df['value'].describe())

print("\nMost common units:")
print(df['unit'].value_counts().head())

Value and Unit extraction complete.

Sample results:


,catalog_content,value,unit
0,Item Name: Gift Basket Village Gourmet Meat an...,7.0,oz
1,"Item Name: NPG Dried Lotus Seeds 16 Oz, Uncook...",16.0,oz
2,Item Name: Annies Homegrown Macaroni and Chees...,6.0,oz
3,Item Name: Bear Creek Country Kitchens Creamy ...,10.1,oz
4,Item Name: Japanese Kelp Kombu Umami Soup Stoc...,4.0,g



Value statistics:
count     79.000000
mean      12.888228
std       23.672600
min        0.000000
25%        2.775000
50%        8.000000
75%       12.250000
max      176.000000
Name: value, dtype: float64

Most common units:
unit
oz    57
      21
g      9
l      5
lb     3
Name: count, dtype: int64


In [ ]:
# Use PaddleOCR for image text extraction
print("Starting OCR text extraction with PaddleOCR...")

df['ocr_text'] = ''
batch_size = 10
for i in range(0, len(df), batch_size):
    batch = df.iloc[i:i+batch_size]
    df.loc[batch.index, 'ocr_text'] = batch['image_link'].apply(extract_text_paddleocr)

print("\nOCR extraction complete.")
print("\nSample OCR results:")
display(df[['image_link', 'ocr_text']].head())

In [31]:
# Function to extract text from image URL using EasyOCR
def extract_text_from_url(url):
    try:
        # Download image
        response = requests.get(url, timeout=10)
        if response.status_code != 200:
            return ''
        
        # Convert to PIL Image
        img = Image.open(BytesIO(response.content))
        
        # Run OCR
        results = reader.readtext(np.array(img))
        
        # Extract text from results
        text = ' '.join([result[1] for result in results])
        
        # Basic cleaning
        text = re.sub(r'\s+', ' ', text).strip()
        return text
    except Exception as e:
        print(f"Error processing URL {url}: {str(e)}")
        return ''

# Process images and extract text
print("Starting OCR text extraction...")

# Try to import notebook tqdm, else fallback to standard tqdm
from tqdm import tqdm as tqdm_bar


# Process images in batches to show progress
df['ocr_text'] = ''
batch_size = 10
for i in tqdm_bar(range(0, len(df), batch_size)):
    batch = df.iloc[i:i+batch_size]
    df.loc[batch.index, 'ocr_text'] = batch['image_link'].apply(extract_text_from_url)

print("\nOCR extraction complete.")
print("\nSample OCR results:")
display(df[['image_link', 'ocr_text']].head())

Starting OCR text extraction...
























100%|██████████| 10/10 [22:57<00:00, 137.78s/it]


OCR extraction complete.

Sample OCR results:


,image_link,ocr_text
0,https://m.media-amazon.com/images/I/91GB1wC6Ob...,@ee OBlend Water Crackers 1 0 SI dclieieus exa...
1,https://m.media-amazon.com/images/I/81VnzF1vkv...,"FreShNESS PrESERVED, Just FOR YOU N P G Lotus ..."
2,https://m.media-amazon.com/images/I/51aCDMHMnI...,70 Annies ORGANIC_ Macaroni & Classic Cheddar ...
3,https://m.media-amazon.com/images/I/71dzRyLGPi...,8 BF1NE SEARCREEK SERVINGS Taste! COUNTRY KITC...
4,https://m.media-amazon.com/images/I/71Yu21cGwr...,17+ p 3h lit h @lipgl) 4 gx10gx M mh 4 0 € @it...


In [39]:
# Handle missing values
print("Missing values before cleaning:")
print(df.isnull().sum())

# Fill missing values
df['value'] = df['value'].fillna(0)
df['unit'] = df['unit'].fillna('')
df['ocr_text'] = df['ocr_text'].fillna('')

print("\nMissing values after cleaning:")
print(df.isnull().sum())

# Create final DataFrame with selected columns
final_df = df[[
    'sample_id',
    'value',
    'unit',
    'image_link',
    'ocr_text',
]].copy()

print("\nFinal DataFrame shape:", final_df.shape)
print("\nSample of final DataFrame:")
display(final_df.head())

Missing values before cleaning:
sample_id          0
catalog_content    0
image_link         0
value              0
unit               0
ocr_text           0
dtype: int64

Missing values after cleaning:
sample_id          0
catalog_content    0
image_link         0
value              0
unit               0
ocr_text           0
dtype: int64

Final DataFrame shape: (100, 5)

Sample of final DataFrame:


,sample_id,value,unit,image_link,ocr_text
0,217392,7.0,oz,https://m.media-amazon.com/images/I/91GB1wC6Ob...,@ee OBlend Water Crackers 1 0 SI dclieieus exa...
1,209156,16.0,oz,https://m.media-amazon.com/images/I/81VnzF1vkv...,"FreShNESS PrESERVED, Just FOR YOU N P G Lotus ..."
2,262333,6.0,oz,https://m.media-amazon.com/images/I/51aCDMHMnI...,70 Annies ORGANIC_ Macaroni & Classic Cheddar ...
3,295979,10.1,oz,https://m.media-amazon.com/images/I/71dzRyLGPi...,8 BF1NE SEARCREEK SERVINGS Taste! COUNTRY KITC...
4,50604,4.0,g,https://m.media-amazon.com/images/I/71Yu21cGwr...,17+ p 3h lit h @lipgl) 4 gx10gx M mh 4 0 € @it...
